# Diwali Festival Sales Analysis - Phase 3 & 4: Exploratory Analysis & Customer Segmentation

**Objective**: Conduct deep exploratory data analysis across 10 analytical dimensions, investigate customer spending behaviors, and train unsupervised K-Means customer personas.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid')
data_path = os.path.join('..', 'data', 'processed', 'diwali_sales_cleaned.csv')
df = pd.read_csv(data_path)
print(f'Loaded cleaned data: {df.shape[0]} rows, {df.shape[1]} columns')


### 1. Macro KPIs & Overview


In [ ]:
total_revenue = df['Amount'].sum()
total_orders = df['Orders'].sum()
unique_cust = df['User_ID'].nunique()
avg_ticket = df['Amount'].mean()

print(f'Total Gross Revenue: ₹{total_revenue:,.2f}')
print(f'Total Units Ordered: {total_orders:,}')
print(f'Unique Shoppers: {unique_cust:,}')
print(f'Mean Basket Size: ₹{avg_ticket:,.2f}')


### 2. Gender & Age Group Dynamics


In [ ]:
plt.figure(figsize=(12, 5))
age_gender = df.groupby(['Age Group', 'Gender_Desc'])['Amount'].sum().reset_index()
sns.barplot(data=age_gender, x='Age Group', y='Amount', hue='Gender_Desc', palette=['#E67E22', '#2E4053'])
plt.title('Revenue by Age Bracket & Gender (INR)', fontsize=13, fontweight='bold')
plt.ylabel('Total Revenue (INR)')
plt.show()


### 3. Geographic Performance (Top States & Zones)


In [ ]:
plt.figure(figsize=(12, 5))
top_states = df.groupby('State')['Amount'].sum().sort_values(ascending=False).head(10).reset_index()
sns.barplot(data=top_states, y='State', x='Amount', hue='State', palette='Oranges_r', legend=False)
plt.title('Top 10 States by Diwali Turnover', fontsize=13, fontweight='bold')
plt.xlabel('Revenue (INR)')
plt.show()


### 4. Product Category Performance


In [ ]:
plt.figure(figsize=(12, 5))
top_cats = df.groupby('Product_Category')['Amount'].sum().sort_values(ascending=False).head(10).reset_index()
sns.barplot(data=top_cats, y='Product_Category', x='Amount', hue='Product_Category', palette='YlOrBr_r', legend=False)
plt.title('Top 10 Product Categories by Revenue', fontsize=13, fontweight='bold')
plt.xlabel('Revenue (INR)')
plt.show()


### 5. Customer Persona Clustering (K-Means)
We aggregate transactions to customer profile level (`User_ID`): Total Spend, Total Orders, Transactions Count, and Average Age.


In [ ]:
cust_df = df.groupby('User_ID').agg(
    Total_Spend=('Amount', 'sum'),
    Total_Orders=('Orders', 'sum'),
    Transactions_Count=('Orders', 'count'),
    Avg_Age=('Age', 'mean')
).reset_index()

features = ['Total_Spend', 'Total_Orders', 'Transactions_Count', 'Avg_Age']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cust_df[features])

for k in [3, 4, 5]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    print(f'k={k} -> Inertia: {km.inertia_:.1f}, Silhouette: {sil:.3f}')


In [ ]:
# Train final k=4 model and profile personas
km_final = KMeans(n_clusters=4, random_state=42, n_init=15)
cust_df['Cluster'] = km_final.fit_predict(X_scaled)
profile = cust_df.groupby('Cluster').agg(
    Customers=('User_ID', 'count'),
    Avg_Spend=('Total_Spend', 'mean'),
    Avg_Orders=('Total_Orders', 'mean'),
    Mean_Age=('Avg_Age', 'mean')
)
print('--- Customer Segment Profiles ---')
print(profile)
